# Chapter 10 — Selection functions: a controlled experiment

**Scope.** This chapter is entirely simulated. It teaches completeness and population correction;
it is not a model of the DR3 or DR4 catalogue selection function.

**Goal.** Start with a known population, select sources by an astrometric misfit, and see which parts of that
population can be recovered when the selection probability is known.

For the real DR3 astrometric-orbit catalogue, Lam, El-Badry & Simon (2025) combine analytic and empirical
prescriptions for the probability of publication: https://arxiv.org/abs/2411.00654. A misfit threshold alone
omits the model cascade, orbit fitting and publication cuts. A real-survey benchmark incorporating those steps remains future work.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2, ncx2

rng = np.random.default_rng(10)


## 10.1 A selection rule with a known null distribution

Our toy survey makes 60 independent along-scan measurements over 34 months, each with uncertainty 0.3 mas.
A star is placed at the ecliptic pole so the parallax factors trace a circle. We fit two positions, two proper
motions and parallax. The residual sum of squares divided by the noise variance has 55 degrees of freedom
under the single-star model. Select a source if it exceeds the 99.9% null quantile.

This is an illustrative chi-square cut, not Gaia's RUWE threshold. There are no correlated CCDs in this toy.

In [ ]:
N, T, sigma, alpha = 60, 34/12, 0.3, 0.001  # observations, yr, mas, null selection probability
# Hold the sampling fixed, so noise realisations can be compared at exactly the same design matrix.
t = np.sort(rng.uniform(-T/2, T/2, N))
scan = rng.uniform(0, 2*np.pi, N)
sn, cs = np.sin(scan), np.cos(scan)
f_al = np.cos(2*np.pi*t)*sn + np.sin(2*np.pi*t)*cs
D = np.column_stack([sn, cs, f_al, t*sn, t*cs])
Q, _ = np.linalg.qr(D, mode='reduced')        # orthonormal basis for the five fitted columns
residual = lambda values: values - Q @ (Q.T @ values)
nu = N-D.shape[1]
cut = chi2.isf(alpha, df=nu)
null = rng.normal(size=(N, 5000))             # noise already divided by sigma
null_chi2 = np.sum(residual(null)**2, axis=0)
plt.hist(null_chi2, bins=50, density=True, histtype='step', label='5000 simulated single stars')
x = np.linspace(10, 120, 400)
plt.plot(x, chi2.pdf(x, df=nu), 'k--', label=f'chi-square, {nu} degrees of freedom')
plt.axvline(cut, color='C3', label='selection threshold')
plt.xlabel('residual chi-square'); plt.ylabel('probability density'); plt.legend(); plt.show()
# Statistical calibration: mean chi-square/nu = 1, within three standard errors of the known null model.
assert abs(null_chi2.mean()/nu-1) < 3*np.sqrt(2/(nu*null.shape[1]))
print(f'Null selection threshold: chi² = {cut:.2f}, equivalent UWE = {np.sqrt(cut/nu):.3f}')

**Figure 10.1.** The residual chi-square of 5,000 simulated single stars at fixed observation times and scan
angles (blue histogram), the analytic 55-degree-of-freedom density (black dashed), and the 99.9% threshold (red).
Both axes are dimensionless. The mean is checked against its analytic expectation.

The threshold deliberately permits a small number of noise-only selections. A source passing it is a poor
single-star fit; it has not yet been established as a binary.

## 10.2 Inject an orbit and measure completeness

Inject 500 circular, face-on binaries at each of 25 logarithmically spaced periods. Every system has
$M_1=1\,M_\odot$, a dark $M_2=0.5\,M_\odot$, and parallax 2 mas; orbital phase is uniform.
Kepler's law gives $a=[(M_1+M_2)P_{\rm yr}^2]^{1/3}$ au, and
$a_0=\varpi a M_2/(M_1+M_2)$ mas. These restrictive choices make a controlled experiment, not a realistic population.

If $s$ is the along-scan orbital signal, its unabsorbed part is $r_s=(I-QQ^T)s$.
The residual chi-square has a noncentral chi-square distribution with
$\lambda=\|r_s/\sigma\|^2$. Thus we can compare measured selection fractions with an independently calculated
probability for the same injected phases, without fitting an empirical completeness curve.

In [ ]:
period_days = np.geomspace(1, 1e5, 25)
n_inject = 500
M1, M2, parallax = 1.0, 0.5, 2.0                 # solar masses, mas
completeness, selected, count_variance = [], [], []
for days in period_days:
    P = days/365.25
    a0 = parallax*((M1+M2)*P**2)**(1/3)*M2/(M1+M2)
    phase = rng.uniform(0, 2*np.pi, n_inject)
    angle = 2*np.pi*t[:, None]/P + phase[None, :] # one column per injected system
    signal = a0*(np.cos(angle)*sn[:, None] + np.sin(angle)*cs[:, None])
    r_signal = residual(signal/sigma)
    noncentrality = np.sum(r_signal**2, axis=0)
    prob = ncx2.sf(cut, df=nu, nc=noncentrality)
    noisy = r_signal + residual(rng.normal(size=signal.shape))
    detected = np.sum(noisy**2, axis=0) > cut
    completeness.append(prob.mean())
    selected.append(detected.sum())
    # The phases have different selection probabilities: add their Bernoulli variances.
    count_variance.append(np.sum(prob*(1-prob)))
completeness = np.array(completeness)
selected, count_variance = np.array(selected), np.array(count_variance)
plt.semilogx(period_days, completeness, '-', label='noncentral chi-square expectation')
plt.errorbar(period_days, selected/n_inject, np.sqrt(count_variance)/n_inject, fmt='o',
             label='injections, ±1 standard error')
plt.axvline(T*365.25, color='k', ls=':', label='observing baseline')
plt.xlabel('orbital period [days]'); plt.ylabel('selection probability')
plt.ylim(-0.03, 1.05); plt.legend(fontsize=8); plt.show()
# Calibration against the analytic probabilities: four standard errors plus one discrete count, per period.
# This tolerance is specified before running the experiment, including bins with probabilities near 0 or 1.
assert np.all(np.abs(selected-n_inject*completeness) <= 4*np.sqrt(count_variance)+1)
print('Largest discrepancy [counts]:', np.max(np.abs(selected-n_inject*completeness)))

**Figure 10.2.** Selection probability against period in days for the specified circular, face-on binary
population and the same fixed toy scanning pattern. Blue: analytic probability averaged over the injected phases.
Orange: fractions selected from 500 noisy injections per period, with one-standard-error bars. The dotted line
marks the observing baseline. The assertion checks every period against the known noise model.

Short-period systems have small photocentre orbits at fixed masses. At long periods the five-parameter fit
absorbs much of the slow motion. The selection is therefore not uniform in period. This plot applies to these
masses, distance, orientation, noise and observing times; it cannot be applied as a correction to Gaia's NSS table.

## 10.3 Recover only where the selection probability is usable

The input contains exactly 500 systems at each trial period. Raw selected counts need not be flat.
If a bin has known completeness $c$, the inverse-probability estimate of its input count is $N_{\rm selected}/c$.
We display that correction only for $c\geq0.1$, a threshold chosen before the run to avoid very large weights.
This is a pedagogical stability cut, not a general prescription for catalogue analysis.

In [ ]:
usable = completeness >= 0.1
corrected = selected[usable]/completeness[usable]
uncertainty = np.sqrt(count_variance[usable])/completeness[usable]
plt.semilogx(period_days, np.full(len(period_days), n_inject), 'k--', label='known input count per trial period')
plt.plot(period_days, selected, 'o-', color='C1', label='selected counts')
plt.errorbar(period_days[usable], corrected, uncertainty, fmt='s', color='C2',
             label='inverse-probability counts (c ≥ 0.1)')
plt.xlabel('orbital period [days]'); plt.ylabel('systems per trial period')
plt.legend(fontsize=8); plt.show()
# Recovery of the known injected count. This restates the previous calibration in population-count units;
# it is not a second independent validation of the completeness model.
assert np.all(np.abs(corrected-n_inject) <= 4*uncertainty + 1/completeness[usable])
assert usable.any() and (~usable).any()             # the example contains both recoverable and omitted regions


**Figure 10.3.** Known input counts (black dashed), selected counts (orange), and inverse-probability counts
where completeness is at least 0.1 (green squares with one-standard-error bars), against period in days.
Each point represents one trial period, not an observed Gaia period bin.

Where the selection probability is known and reasonably large, the correction can recover the injected counts
within their sampling errors. Missing green points are deliberately left uncorrected. A zero or poorly known
selection probability cannot be repaired by assigning larger weights. Uncertainty in a real selection function
must also be propagated; our analytic probabilities were supplied by the known injection model.



## 10.4 Check on a separate population—and change the inclinations

Figures 10.2–10.3 reuse one injection set. Now use independent calibration and test samples, both with
isotropic orbital poles: $\cos i$ is uniform from −1 to 1, the node and phase are uniform from 0 to $2\pi$.
The orbit is still circular, with the same masses, distance and scanning pattern. Rotate the projected ellipse
by the node before taking its along-scan component.

At each period, average analytic detection probabilities over 4,000 calibration systems. Then simulate noisy
data for 1,000 **new** systems. Correct their selected count twice: once with the old face-on completeness,
and once with the isotropic calibration. For the matched correction, propagate binomial test-count variance
and the Monte Carlo uncertainty in the calibration mean. These errors use a first-order propagation, suitable
for the displayed bins with completeness at least 0.1. The test population is independent of both calibrations.


In [ ]:
def orbit_residuals(days, size, generator):
    P = days/365.25
    a0 = parallax*((M1+M2)*P**2)**(1/3)*M2/(M1+M2)
    angle = 2*np.pi*t[:, None]/P + generator.uniform(0, 2*np.pi, size)
    cosi = generator.uniform(-1, 1, size)        # isotropic poles, not uniform inclination
    node = generator.uniform(0, 2*np.pi, size)
    x, y = a0*np.cos(angle), a0*cosi*np.sin(angle)
    east = x*np.cos(node)-y*np.sin(node)
    north = x*np.sin(node)+y*np.cos(node)
    return residual((east*sn[:, None]+north*cs[:, None])/sigma)

cal_rng, test_rng = np.random.default_rng(101), np.random.default_rng(102)
n_cal, n_test = 4000, 1000
iso_c, iso_var, test_count = [], [], []
for days in period_days:
    r_cal = orbit_residuals(days, n_cal, cal_rng)
    p_cal = ncx2.sf(cut, df=nu, nc=np.sum(r_cal**2, axis=0))
    iso_c.append(p_cal.mean())
    iso_var.append(p_cal.var(ddof=1)/n_cal)     # variance of the calibration mean
    r_test = orbit_residuals(days, n_test, test_rng)
    noisy_test = r_test + residual(test_rng.normal(size=r_test.shape))
    test_count.append(np.sum(np.sum(noisy_test**2, axis=0) > cut))
iso_c, iso_var, test_count = map(np.array, (iso_c, iso_var, test_count))
valid = (iso_c >= 0.1) & (completeness >= 0.1)
recovered = test_count[valid]/iso_c[valid]
# Var(K/c) ≈ N(1-c)/c + N² Var(c)/c² for independent count and calibration.
err = np.sqrt(n_test*(1-iso_c[valid])/iso_c[valid] + n_test**2*iso_var[valid]/iso_c[valid]**2)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogx(period_days, completeness, label='face-on calibration')
ax[0].plot(period_days, iso_c, label='isotropic calibration')
ax[0].set(xlabel='orbital period [days]', ylabel='selection probability', ylim=(-0.03, 1.05))
ax[0].legend(fontsize=8)
ax[1].semilogx(period_days[valid], test_count[valid]/completeness[valid], 'o-',
               color='C3', label='isotropic test / face-on calibration')
ax[1].errorbar(period_days[valid], recovered, err, fmt='s', color='C2', label='matched calibration, ±1 SE')
ax[1].axhline(n_test, color='k', ls='--', label='known test population')
ax[1].set(xlabel='orbital period [days]', ylabel='inferred systems per trial period')
ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()
# A held-out recovery check, with a prechosen four-SE tolerance plus one weighted count.
assert valid.any()
assert np.all(np.abs(recovered-n_test) <= 4*err + 1/iso_c[valid])


**Figure 10.4.** Left: analytic selection probability averaged over face-on orbits (blue) and over an
independent isotropic calibration sample (orange), against period in days. Right: counts inferred from the
same independent isotropic test sample using the face-on calibration (red circles) or the matched isotropic
calibration (green squares, one-standard-error bars including calibration sampling uncertainty). The black
dashed line is the known 1,000 test systems at each period. Only periods where both completeness estimates
are at least 0.1 are shown on the right.

Changing the inclination distribution changes the calibration even at fixed masses and distance. A correction
can pass its own injection test and still misrepresent a different population. This held-out test isolates that
problem under a known toy survey; it does not validate a Gaia selection function or solve the inverse problem
when the population's orientations, masses or luminosity ratios are unknown.


## From this experiment to a real catalogue

Reproduce a published DR3 selection-function benchmark with matched assumptions, including model selection
and publication cuts. That extension remains future work; it is outside this controlled experiment. The DR4 comparison awaits public data.

**Exercise.** Change the test population’s distance while keeping the calibration fixed. Explain the resulting bias.

## References

- Lam, El-Badry & Simon (2025), *A Fast, Analytic Empirical Model of the Gaia Data Release 3 Astrometric Orbit Catalog Selection Function*, ApJ 987, 215: https://arxiv.org/abs/2411.00654 ; https://doi.org/10.3847/1538-4357/addac2
- Halbwachs et al. (2023), *Gaia Data Release 3. Astrometric binary star processing*, A&A 674, A9: https://arxiv.org/abs/2206.05726